In [68]:
!pip install haversine

Defaulting to user installation because normal site-packages is not writeable


In [69]:
# Import Necessary Libraries
import pandas as pd
import numpy as np
from haversine import haversine

In [70]:
# Load the dataset
file_path = "D:\\AIS Project\\datasets\\datasets\\training_set.csv"
df = pd.read_csv(file_path, low_memory=False)

In [71]:
# Drop unnecessary column
if 'Unnamed: 0' in df.columns:
    df.drop(columns=['Unnamed: 0'], inplace=True)

In [72]:
# Convert timestamp to datetime
df['timestamp'] = pd.to_datetime(df['timestamp'], errors='coerce')

In [73]:
# Drop duplicate entries with the same timestamp for the same vessel (mmsi)
df = df.drop_duplicates(subset=['timestamp', 'mmsi'])

In [74]:
# Drop rows with missing latitude and longitude values
df = df.dropna(subset=['lat', 'lon'])

In [75]:
# Ensure latitude and longitude are within valid ranges
df = df[(df['lat'].between(-90, 90)) & (df['lon'].between(-180, 180))]

In [76]:
# Remove rows with invalid or negative speed
df = df[df['speed'] >= 0]

In [77]:
# Create 'Is_Moving' column based on speed
df['Is_Moving'] = df['speed'] > 0

# Group by MMSI to handle each vessel separately
voyages = []
for mmsi, group in df.groupby('mmsi'):
    # Sort each group chronologically by timestamp
    group = group.sort_values('timestamp').reset_index(drop=True)
    # Assign Voyage IDs based on changes in moving state
    group['Voyage_ID'] = (group['Is_Moving'] != group['Is_Moving'].shift()).cumsum()
    voyages.append(group)

In [78]:
# Concatenate all processed groups
processed_data = pd.concat(voyages, ignore_index=True)

In [79]:
# Save the processed data
output_file_path = "D:\\AIS Project\\datasets\\filtered_voyage_data.csv"
processed_data.to_csv(output_file_path, index=False)

print(f"Processed data saved to {output_file_path}")

Processed data saved to D:\AIS Project\datasets\filtered_voyage_data.csv


In [80]:
# Load the filtered voyage data
file_path = "D:\\AIS Project\\datasets\\filtered_voyage_data.csv"
df = pd.read_csv(file_path, low_memory=False)

# Function to calculate Douglas-Peucker simplified distance
def calculate_voyage_distance(voyage_df):
    points = voyage_df[['lat', 'lon']].to_numpy()
    total_distance = 0.0
    for i in range(1, len(points)):
        total_distance += haversine(points[i - 1], points[i])  # Distance between consecutive points
    return total_distance

# Initialize a column to store voyage distances
df['Voyage_Distance'] = np.nan

# Calculate voyage distances for each MMSI and Voyage_ID
for (mmsi, voyage_id), group in df.groupby(['mmsi', 'Voyage_ID']):
    distance = calculate_voyage_distance(group)
    df.loc[group.index, 'Voyage_Distance'] = distance

# Save the updated dataframe to a new file
output_file_path = "D:\\AIS Project\\datasets\\voyage_distances.csv"
df.to_csv(output_file_path, index=False)

print(f"Voyage distances calculated and saved to {output_file_path}")


Voyage distances calculated and saved to D:\AIS Project\datasets\voyage_distances.csv
